# Solution E2 · The ColumnTransformer that silently drops the most important columns

**Type**: Find the bug

## Objective

Find out why a model on Titanic has an AUC of 0.7 instead of 0.85 even though there are no errors: a `ColumnTransformer` with
`remainder="drop"` (the default) dropped `sex`, `title` and `pclass`. Rebuild the lost columns and fix it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
NUM = ["age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

ct_bug = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["age", "siblings_spouses", "parents_children", "fare"]),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), ["port", "deck"]),
], sparse_threshold=0)

## Commented solution

In [ ]:
bug_model = make_pipeline(ct_bug, LogisticRegression(max_iter=2000))
cv_auc_bug = cross_val_score(bug_model, X_tr, y_tr, cv=cv, scoring="roc_auc").mean()
print(f"cross-validated AUC: {cv_auc_bug:.3f}")

**Where the bug is.** The colleague's `ColumnTransformer` lists only six columns. With `remainder="drop"` (default) all the others (`pclass`,
`sex`, `title`) are dropped without warning, and they are exactly the three most predictive features of Titanic: the model works,
but with a third of the information.

In [ ]:
ct_bug.fit(X_tr, y_tr)
name, _, indices = ct_bug.transformers_[-1]
lost_columns = sorted(X_tr.columns[indices])
print("last entry of transformers_:", name, "| dropped columns:", lost_columns)

In [ ]:
ct_ok = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["age", "siblings_spouses", "parents_children", "fare"]),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), ["port", "deck", "sex", "title"]),
    ("pclass", "passthrough", ["pclass"]),
], sparse_threshold=0)
cv_auc_ok = cross_val_score(make_pipeline(ct_ok, LogisticRegression(max_iter=2000)), X_tr, y_tr, cv=cv, scoring="roc_auc").mean()
print(f"cross-validated AUC: bug {cv_auc_bug:.3f} | fixed {cv_auc_ok:.3f}")

In [ ]:
def check_columns(ct, X):
    name, _, indices = ct.transformers_[-1]
    if name == "remainder" and len(indices) > 0:
        raise ValueError(f"columns silently dropped: {list(X.columns[indices])}")

## Check

In [ ]:
with verify():
    check(cv_auc_bug < 0.78, "the colleague's model is far below expectations (AUC below 0.78)", "don't modify the colleague's code")
    check(lost_columns == ["pclass", "sex", "title"], "dropped columns: pclass, sex, title", "indices in ct_bug.transformers_[-1][2], names with X.columns[indices]")
    check(cv_auc_ok > cv_auc_bug + 0.08 and cv_auc_ok > 0.82, "including the lost columns the AUC goes up by more than 0.08 (above 0.82)",
          "add sex and title to the categoricals and pclass in passthrough")
    try:
        check_columns(ct_bug, X_tr)
        raises_bug = False
    except ValueError:
        raises_bug = True
    ct_ok.fit(X_tr, y_tr)
    try:
        check_columns(ct_ok, X_tr)
        raises_ok = False
    except ValueError:
        raises_ok = True
    check(raises_bug and not raises_ok, "check_columns raises for the colleague's preprocessor and not for the correct one",
          "raise ValueError if ct.transformers_[-1][0] == 'remainder' and the list of indices is not empty")
done()

## What you learned

- `ColumnTransformer` has `remainder="drop"` by default: every column not listed **disappears**, with no error or warning. The colleague's
  AUC (about 0.69) is plausible and stays in a range where nobody gets alarmed.
- The defense is explicit: (1) list all the columns, or `remainder="passthrough"` if you know they are already numeric; (2) inspect
  `transformers_` or `get_feature_names_out()` after the fit; (3) put an automatic check (like `check_columns`) in the
  training code, which fails loudly when the schema changes.
- In production the risk is double: if a new column arrives and is not listed, the model ignores it without saying so. A check
  on the expected input columns (`feature_names_in_`) is part of the pipeline of a serious service.

## Common pitfall

Thinking that "if a column is not needed, I don't list it": fine, but then dropping it must be a written decision
(explicit `remainder="drop"`, maybe with a comment), not a side effect of a default.

## Interview question

<details><summary>How do you notice that a Pipeline with ColumnTransformer is ignoring an important column?</summary>

I look at the output names (`get_feature_names_out()`) and `transformers_`: the `remainder` entry with non-empty indices says which columns were
dropped. In the training code I add a check that fails if the remainder is not empty, and in the service I verify
the input columns against `feature_names_in_`. An AUC lower than expected with no errors is often the first signal.

</details>